In [29]:
import sys
import random
import math
import logging
import json
import os

logging.basicConfig(
    filename = "game.log",
    level = logging.INFO,
    format =  "%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

class Game:

    def __init__(self):
        self.mode = None
        self.last_guessDiff = None
        self.last_matchDiff = None
        self.state = "menu"
        self.game_properties = {
            "menu": self.menu,
            "mode": self.mode_pick,
            "guess": self.guess_difficulty,
            "match": self.match_difficulty,
            "stats": self.view_stats,
            "guess_easy": lambda: self.guessing_play(1, 10, math.inf),
            "guess_medium": lambda: self.guessing_play(1, 100, 5),
            "guess_hard": lambda: self.guessing_play(1, 1000, 3),
            "match_easy": lambda: self.matching_play(1, 20, math.inf),
            "match_medium": lambda: self.matching_play(1, 100, 5),
            "match_hard": lambda: self.matching_play(1, 1000, 3),
            "continue": self.game_end
        }
        self.stats = {}
        
    def create_stats(self):
        if not os.path.exists("stats.json"):
            with open("stats.json", "w") as file:
                json.dump({
                    "games_played": 0,
                    "wins": 0,
                    "wins_with_attempts": 0,
                    "losses": 0,
                    "games_easy": 0,
                    "easy_wins": 0,
                    "games_medium": 0,
                    "medium_wins": 0,
                    "medium_wins_with_attempts": 0,
                    "medium_losses": 0,
                    "games_hard": 0,
                    "hard_wins": 0,
                    "hard_wins_with_attempts": 0,
                    "hard_losses": 0,
                }, file)

    def load_stats(self):
        with open("stats.json", "r") as file:
            self.stats = json.load(file)

    def save_stats(self):
        with open("stats.json", "w") as file:
            json.dump(self.stats, file)

    def run(self):
        self.create_stats()
        self.load_stats()
        logger.info("Game started")
        while self.state:
            self.state = self.game_properties[self.state]()
            
    def menu(self):
        print("-" * 50)
        print("[Play]\n[View Stats]\n[Exit]")
        print("-" * 50)
        while True:
            menu_input = input().lower()
            if menu_input in {"play"}:
                return "mode"
            elif menu_input in {"view stats"}:
                return "stats"
            elif menu_input in {"exit"}:
                print("-" * 50)
                print("Thx for playing!")
                print("-" * 50)
                logger.info("Player exited the game")
                logging.shutdown()
                sys.exit()
            else:
                print("-" * 50)
                print("Invalid command!")
                print("-" * 50)

    def view_stats(self):
        win_rate = round((self.stats["wins"] / self.stats["games_played"]) * 100, 2)
        all_stats = [f"Games: {self.stats['games_played']}",
                    "-" * 50,
                    f"Wins: {self.stats['wins']}",
                    "-" * 50,
                    f"Wins with attempts left: {self.stats['wins_with_attempts']}",
                    "-" * 50,
                    f"Losses: {self.stats['losses']}",
                    "-" * 50,
                    f"Win rate: {win_rate}%",
                    "-" * 50,
                    f"Games played on difficulty: ",
                    "-" * 50,
                    f"Easy: {self.stats['games_easy']}",
                    "-" * 50,
                    f"Medium: {self.stats['games_medium']}",
                    "-" * 50,
                    f"Hard: {self.stats['games_hard']}",
                    "-" * 50,
                    f"Wins on difficulty: ",
                    "-" * 50,
                    f"Easy: {self.stats['easy_wins']}",
                    "-" * 50,
                    f"Medium: {self.stats['medium_wins']}",
                    "-" * 50,
                    f"Hard: {self.stats['hard_wins']}",
                    "-" * 50,
                    f"Wins with attempts on difficulty: ",
                    "-" * 50,
                    f"Medium: {self.stats['medium_wins_with_attempts']}",
                    "-" * 50,
                    f"Hard: {self.stats['hard_wins_with_attempts']}",
                    "-" * 50,
                    f"Losses on difficulty: ",
                    "-" * 50,
                    f"Medium: {self.stats['medium_losses']}",
                    "-" * 50,
                    f"Hard: {self.stats['hard_losses']}",
                    "-" * 50]
        print("-" * 50)
        print("\n".join(all_stats))
        print("[Back]")
        print("-" * 50)
        while True:
            back_input = input().lower()
            if back_input in {"back"}:
                return "menu"
            else:
                print("-" * 50)
                print("Invalid command!")
                print("-" * 50)

    def mode_pick(self):
        print("-" * 50)
        print("[Guess]\n[Match]\n[Return]")
        print("-" * 50)
        while True:
            mode_input = input().lower()
            if mode_input in {"guess"}:
                logger.info("Mode selected: Guess")
                return "guess"
            elif mode_input in {"match"}:
                logger.info("Mode selected: Match")
                return "match"
            elif mode_input in {"return"}:
                logger.info("Player returned to menu")
                return "menu"
            else:
                print("Invalid command!")
    
    def guess_difficulty(self):
        print("-" * 50)
        print("[Easy]\n[Medium]\n[Hard]\n[Return]")
        print("-" * 50)
        while True:
            difficulty_input = input().lower()
            if difficulty_input in {"easy"}:
                self.last_guessDiff = "guess_easy"
                logger.info("Difficulty selected: easy")
                return "guess_easy"
            elif difficulty_input in {"medium"}:
                self.last_guessDiff = "guess_medium"
                logger.info("Difficulty selected: medium")
                return "guess_medium"
            elif difficulty_input in {"hard"}:
                self.last_guessDiff = "guess_hard"
                logger.info("Difficulty selected: hard")
                return "guess_hard"
            elif difficulty_input in {"return"}:
                logger.info("Player returned to menu")
                return "mode"
            else:
                print("Invalid choice!")

    def match_difficulty(self):
        print("-" * 50)
        print("[Easy]\n[Medium]\n[Hard]\n[Return]")
        print("-" * 50)
        while True:
            difficulty_input = input().lower()
            if difficulty_input in {"easy"}:
                self.last_matchDiff = "match_easy"
                logger.info("Difficulty selected: easy")
                return "match_easy"
            elif difficulty_input in {"medium"}:
                self.last_matchDiff = "match_medium"
                logger.info("Difficulty selected: medium")
                return "match_medium"
            elif difficulty_input in {"hard"}:
                self.last_matchDiff = "match_hard"
                logger.info("Difficulty selected: hard")
                return "match_hard"
            elif difficulty_input in {"return"}:
                logger.info("Returned to menu")
                return "mode"
            else:
                print("Invalid choice!")
                
    def game_prompts(self, random_number, guess):
        difference = abs(random_number - guess)
        if (25 >= difference > 15):
            print("Cold!")
        elif (100 >= difference > 25):
            print("Super cold!")
        elif (difference > 100):
            print("Extremely cold!")
        elif (15 >= difference > 10):
            print("Warm!")
        elif (self.last_guessDiff != "guess_easy" and 10 >= difference > 5):
            print("Very warm!")
        elif (self.last_guessDiff != "guess_easy" and 5 >= difference > 0):
            print("Really hot!")

    def stat_check(self, diff, attempts, result):
        self.stats["games_played"] += 1
        if (diff == "guess_easy" and result == "win"):
            self.stats["games_easy"] += 1
            self.stats["wins"] += 1
            self.stats["easy_wins"] += 1
        elif (diff == "guess_medium" and result == "win"):
            self.stats["games_medium"] += 1
            self.stats["wins"] += 1
            self.stats["medium_wins"] += 1
            if (attempts > 0 and attempts != math.inf):
                self.stats["wins_with_attempts"] += 1
                self.stats["medium_wins_with_attempts"] += 1
        elif (diff == "guess_hard" and result == "win"):
            self.stats["games_hard"] += 1 
            self.stats["wins"] += 1
            self.stats["hard_wins"] += 1
            if (attempts > 0 and attempts != math.inf):
                self.stats["wins_with_attempts"] += 1
                self.stats["hard_wins_with_attempts"] += 1

        if (diff == "guess_medium" and result == "loss"):
            self.stats["games_medium"] += 1
            self.stats["losses"] += 1
            self.stats["medium_losses"] += 1
        elif (diff == "guess_hard" and result == "loss"):
            self.stats["games_hard"] += 1 
            self.stats["losses"] += 1
            self.stats["hard_losses"] += 1
            
    def guessing_play(self, min_range, max_range, attempts_limit):
        random_number = random.randint(min_range, max_range)
        print("-" * 50)
        print("Guess!")
        while True:
            try:
                guess = int(input())
                    
                if (guess > random_number):
                    print("-" * 50)
                    print ("Too high!")
                    self.game_prompts(random_number, guess)
                    print("-" * 50)
                    attempts_limit -= 1
                elif (guess < random_number):
                    print("-" * 50)
                    print ("Too low!")
                    self.game_prompts(random_number, guess)
                    print("-" * 50)
                    attempts_limit -= 1
                elif (guess == random_number):
                    self.stat_check(self.last_guessDiff, attempts_limit, "win")
                    self.save_stats()
                    logger.info(
                        f"Win | difficulty = {self.last_guessDiff} | number = {random_number}"
                    )
                    print("-" * 50)
                    print(f"Great guess!\nTry again?\nYes/No")
                    print("-" * 50)
                    self.mode = "guessed"
                    return "continue"
                        
                if (attempts_limit != math.inf and attempts_limit > 0):
                    print(f"Attempts left: {attempts_limit}")
                    print("-" * 50)
                    continue
                elif (attempts_limit == 0):
                    self.stat_check(self.last_guessDiff, attempts_limit, "loss")
                    self.save_stats()
                    logger.info(
                        f"Loss | difficulty = {self.last_guessDiff} | number = {random_number}"
                    )
                    print(f"No attempts left!\nThe number was {random_number}\nTry again?\nYes/No")
                    print("-" * 50)
                    self.mode = "guessed"
                    return "continue"
                        
            except ValueError:
                print("-" * 50)
                print("Invalid guess!")
                print("-" * 50)

    def matching_play(self, min_range, max_range, attempts_limit):
        row_one = []
        row_two = []
        row_three = []
        row_four = []
        matching_rows = {
            "first_row": {
                1: 0, 2: 0, 3: 0, 4: 0
            },
            "second_row": {
                1: 0, 2: 0, 3: 0, 4: 0
            },
            "third_row": {
                1: 0, 2: 0, 3: 0, 4: 0
            },
            "fourth_row": {
                1: 0, 2: 0, 3: 0, 4: 0
            }
        }
        matched_numbers = {
            "first_row": {
                1: "XXX", 2: "XXX", 3: "XXX", 4: "XXX"
            },
            "second_row": {
                1: "XXX", 2: "XXX", 3: "XXX", 4: "XXX"
            },
            "third_row": {
                1: "XXX", 2: "XXX", 3: "XXX", 4: "XXX"
            },
            "fourth_row": {
                1: "XXX", 2: "XXX", 3: "XXX", 4: "XXX"
            }
        }
        while True:
            for row in matched_numbers["first_row"]:
                row_one.append(matched_numbers["first_row"][row])
            for row in matched_numbers["second_row"]:
                row_two.append(matched_numbers["second_row"][row])
            for row in matched_numbers["third_row"]:
                row_three.append(matched_numbers["third_row"][row])
            for row in matched_numbers["fourth_row"]:
                row_four.append(matched_numbers["fourth_row"][row])
            break
        print("-" * 50)
        print(f"{"|".join(row_one)}\n{"|".join(row_two)}\n{"|".join(row_three)}\n{"|".join(row_four)}")
        print("-" * 50)
        row_map = {
            1: "first_row",
            2: "second_row",
            3: "third_row",
            4: "fourth_row"
        }
        get_key = None
        number_generator = random.randint(min_range, max_range)
        exclude_generator = []
        exclude_generator.append(number_generator)
        counter = 1
        while counter < 8:
            number_generator = random.randint(min_range, max_range)
            if number_generator not in exclude_generator:
                exclude_generator.append(number_generator)
                counter += 1
        for number in exclude_generator:
            placed = 0
            while placed < 2:
                row = random.randint(1,4)
                col = random.randint(1,4)
                target_row = row_map[row]
                if matching_rows[target_row][col] == 0:
                    matching_rows[target_row][col] = number
                    placed += 1
        while matched_numbers != matching_rows:
            try:
                match_input = int(input())
                for row_key, row_value in matching_rows.items():
                    for col_key, col_value in matching_rows[row_key].items():
                        correct_key = col_key - 1
                        if row_key == "first_row":
                            if match_input == col_value:
                                matched_numbers["first_row"][col_key] = col_value
                                row_one[correct_key] = col_value
                        if row_key == "second_row":
                            if match_input == col_value:
                                matched_numbers["second_row"][col_key] = col_value
                                row_two[correct_key] = col_value
                        if row_key == "third_row":
                            if match_input == col_value:
                                matched_numbers["third_row"][col_key] = col_value
                                row_three[correct_key] = col_value
                        if row_key == "fourth_row":
                            if match_input == col_value:
                                matched_numbers["fourth_row"][col_key] = col_value
                                row_four[correct_key] = col_value
                print("-" * 50)
                print(f"{"|".join(str(v) for v in row_one)}\n"
                f"{"|".join(str(v) for v in row_two)}\n"
                f"{"|".join(str(v) for v in row_three)}\n"
                f"{"|".join(str(v) for v in row_four)}")
                print("-" * 50)
                if matched_numbers == matching_rows:
                    print("-" * 50)
                    print(f"Great job!\nTry again?\nYes/No")
                    print("-" * 50)
                    self.mode = "matched"
                    return "continue"
            except ValueError:
                print("-" * 50)
                print("Invalid number!")
                print("-" * 50)

    def game_end(self):
        while True:
            choice = input().lower()
            if (self.mode == "guessed"):
                if (choice == "yes"):
                    logger.info("Player started a new game on the same difficulty")
                    return self.last_guessDiff
                elif (choice == "no"):
                    logger.info("Player ended the game")
                    return "menu"
                else:
                    print("-" * 50)
                    print("Invalid command!")
                    print("Try again?\nYes/No")
                    print("-" * 50)
                    
            elif (self.mode == "matched"):
                if (choice == "yes"):
                    logger.info("Player started a new game on the same difficulty")
                    return self.last_matchDiff
                elif (choice == "no"):
                    logger.info("Player ended the game")
                    return "menu"
                else:
                    print("-" * 50)
                    print("Invalid command!")
                    print("Try again?\nYes/No")
                    print("-" * 50)
                    
game = Game()
game.run()

--------------------------------------------------
[Play]
[View Stats]
[Exit]
--------------------------------------------------


 play


--------------------------------------------------
[Guess]
[Match]
[Return]
--------------------------------------------------


 match


--------------------------------------------------
[Easy]
[Medium]
[Hard]
[Return]
--------------------------------------------------


...

 no


--------------------------------------------------
[Play]
[View Stats]
[Exit]
--------------------------------------------------


 exit


--------------------------------------------------
Thx for playing!
--------------------------------------------------


SystemExit: 

In [310]:
'''
added mode menu
added another mode
added a check in the end of the game to either return to the menu or return to the correct game mode
'''
#add stats for new mode, fix difficulties for new mode

'\nadded mode menu\nadded another mode\nadded a check in the end of the game to either return to the menu or return to the correct game mode\n'

In [ ]:
import unittest
from unittest.mock import patch

class TestGame(unittest.TestCase):

    @patch("builtins.input", side_effect = ["asd", "213","play"])
    def test_menu_play(self, mock_input):
        game = Game()
        result = game.menu()
        self.assertEqual(result, "difficulty")

    def test_menu_exit(self):
        game = Game()
        with patch("builtins.input", side_effect = ["sdsa", "321", "exit"]):
            with self.assertRaises(SystemExit):
                game.menu()

    @patch("builtins.input", side_effect = ["asd", "123", "return"])
    def test_return_invalid_then_correct(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "menu")
    
    @patch("builtins.input", return_value = "easy")
    def test_difficulty_easy(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "easy")
        self.assertEqual(game.last_guessDiff, "easy")

    @patch("builtins.input", side_effect = ["wrong", "medium"])
    def test_invalid_then_medium(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "medium")
        self.assertEqual(game.last_guessDiff, "medium")

    @patch("random.randint", return_value = 7)
    @patch("builtins.input", side_effect = ["2", "3", "9", "7"])
    def test_guessing_play_win_on_correct_guess(self, mock_input, mock_randint):
        game = Game()
        result = game.guessing_play(1, 10, math.inf)
        self.assertEqual(result, "continue")

    @patch("builtins.input", side_effect = ["asdas", "wrong", "hard"])
    def test_invalid_difficulty_then_hard(self, mock_input):
        game = Game()
        result = game.difficulty()
        self.assertEqual(result, "hard")
        self.assertEqual(game.last_guessDiff, "hard")

    @patch("random.randint", return_value = 44)
    @patch("builtins.input", side_effect = ["2", "45", "60", "30", "66"])
    def test_exceeding_attempt_limits(self, mock_input, mock_randint):
        game = Game()
        result = game.guessing_play(1, 100, 5)
        self.assertEqual(result, "continue")

    @patch("builtins.input", side_effect = ["sa", "123", "no"])
    def test_invalid_game_end_input_return(self, mock_input):
        game = Game()
        result = game.game_end()
        self.assertEqual(result, "menu")

    @patch("builtins.input", side_effect = ["sda", "213", "yes"])
    def test_invalid_game_end_input_continue_last_guessDiff(self, mock_input):
        game = Game()
        game.last_guessDiff = "hard"
        result = game.game_end()
        self.assertEqual(result, "hard")

In [ ]:
#@patch - switches given part in the code
#"builtins.input" - switches input in the code
#"random.randint" - switches the random_number in the code
#side_effect - used for multiple inputs
#return_value = "easy" - the input that "builtins.input" switches, used for a value that is given once
#mock_input - is just needed to be there
#result = takes the real value from the code
#self.assertEqual(result, "easy") - compares result to the given string - "easy" if it doesnt match an error is given
#self.assertEqual(game.last_guessDiff, "easy") - checks the the code's memory variable

In [ ]:
suite = unittest.TestLoader().loadTestsFromTestCase(TestGame)
unittest.TextTestRunner(verbosity = 2).run(suite)